Data Acquisition and Leaf-grouped Split.

Everything below (folder names, per-class counts, leaf_grouping coverage) was checked directly against spMohanty/PlantVillage-Dataset on GitHub 

This replaces a Hugging Face "mohanty/PlantVillage" load because color, grayscale, segmented were bundled together, which is both the wrong label granularity and a leakage risk (the same leaf's three representations could land in different splits). 
Going straight to the source repo avoids both.

I summarized what code snippets do into steps.

In [1]:

import csv
import random
import subprocess
from collections import Counter, defaultdict
from pathlib import Path

REPO = "https://github.com/spMohanty/PlantVillage-Dataset.git"
DEST = Path("PlantVillage-Dataset")
SEED = 42


Folder names and image counts (raw/color)

In [2]:

CLASSES = {
    "Tomato___Bacterial_spot": 2127,
    "Tomato___Early_blight": 1000,
    "Tomato___Late_blight": 1909,
    "Tomato___Leaf_Mold": 952,
    "Tomato___Septoria_leaf_spot": 1771,
    "Tomato___Spider_mites Two-spotted_spider_mite": 1676,
    "Tomato___Target_Spot": 1404,
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": 5357,
    "Tomato___Tomato_mosaic_virus": 373,   # no leaf-group CSV
    "Tomato___healthy": 1591,
    "Potato___Early_blight": 1000,
    "Potato___Late_blight": 1000,
    "Potato___healthy": 152,
    "Pepper,_bell___Bacterial_spot": 997,
    "Pepper,_bell___healthy": 1478,
}


Step 1. Sparse checkout (pulling the only folders listed above and not the image repo)

In [3]:

paths = [f"raw/color/{c}" for c in CLASSES] + ["leaf_grouping/filtered_leafmaps"]

if not DEST.exists():
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", REPO, str(DEST)], check=True)
    subprocess.run(["git", "sparse-checkout", "init", "--cone"], cwd=DEST, check=True)
    subprocess.run(["git", "sparse-checkout", "set"] + paths, cwd=DEST, check=True)
    subprocess.run(["git", "checkout"], cwd=DEST, check=True)


CalledProcessError: Command '['git', 'sparse-checkout', 'init', '--cone']' returned non-zero exit status 128.

Step 2. Building a table(filepath, class_name, leaf_group). 

* where no leaf-group CSV exists (tomato mosaic virus), each image becomes its own group which is equivalent to a random split for that slice - this is a limitation

In [ ]:

rows = []
for class_name in CLASSES:
    class_dir = DEST / "raw/color" / class_name
    csv_path = DEST / "leaf_grouping/filtered_leafmaps" / f"{class_name}.csv"
    leaf_map = {}
    if csv_path.exists():
        with open(csv_path, newline="", encoding="utf-8-sig") as f:
            for r in csv.DictReader(f):
                leaf_map[r["File Name"]] = r["Leaf #"]
    for img_path in class_dir.glob("*"):
        group = leaf_map.get(img_path.name, f"__nogroup__{img_path.name}")
        rows.append((str(img_path), class_name, group))

print(f"Total images: {len(rows)}")



Total images: 14114


Step 3. Group-aware split, done per class so class balance is preserved across train/val/test. This is not just leakage-safe, but fair for the three-model comparison too.

In [ ]:


random.seed(SEED)
by_class = defaultdict(lambda: defaultdict(list))
for filepath, class_name, group in rows:
    by_class[class_name][group].append(filepath)

train_rows, val_rows, test_rows = [], [], []
for class_name, group_dict in by_class.items():
    keys = list(group_dict.keys())
    random.shuffle(keys)
    n = len(keys)
    t_end, v_end = int(n * 0.70), int(n * 0.85)  # 70/15/15
    for i, k in enumerate(keys):
        bucket = train_rows if i < t_end else (val_rows if i < v_end else test_rows)
        bucket.extend((fp, class_name) for fp in group_dict[k])


Step 4: Report the exact numbers

* random seed, count per split, class distribution per split

* these figures and numbers are needed in the report

In [ ]:

for name, split in [("Train", train_rows), ("Val", val_rows), ("Test", test_rows)]:
    dist = Counter(c for _, c in split)
    print(f"\n{name}: {len(split)} images (seed={SEED})")
    for cname in CLASSES:
        print(f"  {cname}: {dist[cname]}")

# train_rows / val_rows / test_rows are lists of (filepath, class_name).
# Build torch Datasets from these using the transforms in
# preprocessing_augmentation.py (train_transform for train_rows only).


Train: 9875 images (seed=42)
  Tomato___Bacterial_spot: 1488
  Tomato___Early_blight: 700
  Tomato___Late_blight: 1336
  Tomato___Leaf_Mold: 666
  Tomato___Septoria_leaf_spot: 1239
  Tomato___Spider_mites Two-spotted_spider_mite: 1173
  Tomato___Target_Spot: 36
  Tomato___Tomato_Yellow_Leaf_Curl_Virus: 0
  Tomato___Tomato_mosaic_virus: 0
  Tomato___healthy: 0
  Potato___Early_blight: 700
  Potato___Late_blight: 700
  Potato___healthy: 106
  Pepper,_bell___Bacterial_spot: 697
  Pepper,_bell___healthy: 1034

Val: 2118 images (seed=42)
  Tomato___Bacterial_spot: 319
  Tomato___Early_blight: 150
  Tomato___Late_blight: 286
  Tomato___Leaf_Mold: 143
  Tomato___Septoria_leaf_spot: 266
  Tomato___Spider_mites Two-spotted_spider_mite: 251
  Tomato___Target_Spot: 8
  Tomato___Tomato_Yellow_Leaf_Curl_Virus: 0
  Tomato___Tomato_mosaic_virus: 0
  Tomato___healthy: 0
  Potato___Early_blight: 150
  Potato___Late_blight: 150
  Potato___healthy: 23
  Pepper,_bell___Bacterial_spot: 150
  Pepper,_bell_